# ML API Launcher for Colab and VS Code

Run the cells top to bottom.
This notebook installs required runtime packages, imports the existing API and detector modules, and starts the Flask API.

In [2]:
import importlib.util
import subprocess
import sys


REQUIRED_PACKAGES = {
    "flask": "flask",
    "flask_cors": "flask-cors",
    "requests": "requests",
    "pydantic": "pydantic",
}

def ensure_required_packages() -> None:
    missing = []
    for module_name, package_name in REQUIRED_PACKAGES.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(package_name)

    if not missing:
        print("All required runtime packages are already installed.")
        return

    print(f"Installing missing packages: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

ensure_required_packages()

Installing missing packages: flask-cors


In [5]:
import os
from pathlib import Path

candidates = [
    Path.cwd(),
    Path.cwd() / "ml-service",
    Path("/content"),
    Path("/content/ml-service"),
]

workdir = None
for p in candidates:
    if (p / "api.py").exists() and (p / "fraud_detector.py").exists():
        workdir = p
        break

if workdir is None:
    raise FileNotFoundError(
        "Could not find api.py and fraud_detector.py. Place this notebook in ml-service or run from project root."
    )

os.chdir(workdir)
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))

os.environ.setdefault("ML_SERVICE_PORT", "5001")
print(f"Working directory: {workdir}")
print(f"ML_SERVICE_PORT: {os.environ['ML_SERVICE_PORT']}")

FileNotFoundError: Could not find api.py and fraud_detector.py. Place this notebook in ml-service or run from project root.

In [ ]:
from pathlib import Path

mode = "UNKNOWN"
port = int(os.environ.get("ML_SERVICE_PORT", "5001"))

if Path("api.py").exists() and Path("fraud_detector.py").exists():
    from fraud_detector import MOCK_MODE, get_detector
    from api import app

    get_detector()
    mode = "MOCK" if MOCK_MODE else "REAL"
else:
    if not (Path("api.ipynb").exists() and Path("fraud_detector.ipynb").exists()):
        raise FileNotFoundError(
            "Could not find api.py/fraud_detector.py or api.ipynb/fraud_detector.ipynb in this folder."
        )

    ip = get_ipython()
    if ip is None:
        raise RuntimeError("Notebook fallback requires an active IPython kernel.")

    # Execute notebooks to populate symbols in the current kernel.
    ip.run_line_magic("run", "fraud_detector.ipynb")
    ip.run_line_magic("run", "api.ipynb")

    if "app" not in globals():
        raise RuntimeError("api.ipynb did not define a global Flask app variable named app.")

    app = globals()["app"]
    # If MOCK_MODE was defined by fraud_detector.ipynb, use it.
    mode = "MOCK" if bool(globals().get("MOCK_MODE", False)) else "REAL"

print("ML API launcher ready")
print(f"Mode: {mode}")
print(f"Listening on: http://0.0.0.0:{port}")
print(f"Health endpoint: http://0.0.0.0:{port}/health")

In [ ]:
app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False)